In [9]:
import pandas as pd
from utils.notebook_helpers import get_postgres_engine

engine, pg_conf = get_postgres_engine("../configs/database.yaml")
print("Connected to PostgreSQL:", pg_conf["host"], pg_conf["db"])

Connected to PostgreSQL: localhost storage_analytics


In [10]:
curated_df = pd.read_sql_query("SELECT * FROM curated_device_metrics", engine)
anomaly_df = pd.read_sql_query("SELECT * FROM anomaly_events", engine)

curated_df["timestamp"] = pd.to_datetime(curated_df["timestamp"])
anomaly_df["timestamp"] = pd.to_datetime(anomaly_df["timestamp"])

In [11]:
curated_df.groupby("workload_pattern")[[
    "total_iops",
    "avg_latency_ms",
    "saturation_score",
    "merge_efficiency",
    "queue_efficiency",
    "iowait_pressure",
    "svctm_await_ratio",
    "write_amplification",
]].mean().round(4).sort_values("avg_latency_ms", ascending=False)

,total_iops,avg_latency_ms,saturation_score,merge_efficiency,queue_efficiency,iowait_pressure,svctm_await_ratio,write_amplification
workload_pattern,,,,,,,,
latency_sensitive,197.3624,293.9825,103.5495,0.2761,237.8021,7.1491,0.8210,1.8352
saturated,529.4766,2.3972,218.7106,0.1425,514.9082,11.9641,8.3018,1.5248
small_io_pressure,2543.4005,1.7145,75.3543,0.0241,4451.9063,6.0864,0.8264,0.9718
balanced,342.3540,1.0237,19.9003,0.0748,895.2007,0.7028,1.2090,0.8590
burst_io,1865.8909,0.6496,86.1617,0.0237,2213.1114,3.2385,0.3543,0.7894


In [12]:
anomaly_df.groupby("workload_pattern").size().sort_values(ascending=False)

workload_pattern
balanced             3651
latency_sensitive    2779
burst_io             1136
saturated             638
small_io_pressure     219
dtype: int64

In [13]:
anomaly_df["root_cause_hint"].value_counts()

root_cause_hint
Anomalous behavior detected                                     3967
Composite saturation signal indicates elevated device stress    2264
Latency anomaly detected without clear saturation signal        1119
IOPS anomaly indicates bursty load increase                      570
Latency degradation likely under write-heavy pressure            201
Latency spike likely driven by saturation and queue buildup      126
High IOPS with small requests suggests random IO pressure        102
Latency spike observed during read-heavy demand                   74
Name: count, dtype: int64

In [14]:
anomaly_df[anomaly_df["severity"] == "critical"][
    ["device", "timestamp", "metric_name", "severity", "workload_pattern", "root_cause_hint"]
].sort_values("timestamp")

,device,timestamp,metric_name,severity,workload_pattern,root_cause_hint
1639,sda,2026-03-31 04:39:07.808737+00:00,merge_efficiency,critical,balanced,Anomalous behavior detected
717,nvme0n1,2026-03-31 04:39:26.606062+00:00,merge_efficiency,critical,balanced,Anomalous behavior detected
1649,sda,2026-03-31 07:08:54.093592+00:00,merge_efficiency,critical,balanced,Anomalous behavior detected
1650,sda,2026-03-31 07:14:03.828933+00:00,merge_efficiency,critical,balanced,Anomalous behavior detected
2333,sdb,2026-03-31 10:28:50.973773+00:00,saturation_score,critical,saturated,Composite saturation signal indicates elevated...
...,...,...,...,...,...,...
6346,nvme1n1,2026-04-07 21:29:04.201471+00:00,avg_latency_ms,critical,latency_sensitive,Latency anomaly detected without clear saturat...
6347,nvme1n1,2026-04-07 21:34:28.712653+00:00,avg_latency_ms,critical,latency_sensitive,Latency anomaly detected without clear saturat...
6348,nvme1n1,2026-04-07 21:39:34.688312+00:00,avg_latency_ms,critical,latency_sensitive,Latency spike likely driven by saturation and ...
6349,nvme1n1,2026-04-07 21:43:45.681086+00:00,avg_latency_ms,critical,latency_sensitive,Latency anomaly detected without clear saturat...


In [15]:
anomaly_df.groupby(["device", "root_cause_hint"]).size().sort_values(ascending=False).head(20)

device   root_cause_hint                                             
sdb      Anomalous behavior detected                                     895
sda      Anomalous behavior detected                                     815
nvme1n1  Anomalous behavior detected                                     811
nvme0n1  Anomalous behavior detected                                     750
dm-0     Anomalous behavior detected                                     696
sdb      Latency anomaly detected without clear saturation signal        608
         Composite saturation signal indicates elevated device stress    555
dm-0     Composite saturation signal indicates elevated device stress    514
nvme1n1  Composite saturation signal indicates elevated device stress    454
nvme0n1  Composite saturation signal indicates elevated device stress    380
sda      Composite saturation signal indicates elevated device stress    361
sdb      IOPS anomaly indicates bursty load increase                     205
sda   

In [16]:
anomaly_df.groupby(["metric_name", "root_cause_hint"]).size().sort_values(ascending=False).head(25)

metric_name            root_cause_hint                                             
saturation_score       Composite saturation signal indicates elevated device stress    2264
iowait_pressure        Anomalous behavior detected                                     1843
queue_efficiency       Anomalous behavior detected                                     1320
avg_latency_ms         Latency anomaly detected without clear saturation signal        1119
total_iops             IOPS anomaly indicates bursty load increase                      570
merge_efficiency       Anomalous behavior detected                                      551
total_throughput_mb_s  Anomalous behavior detected                                      253
avg_latency_ms         Latency degradation likely under write-heavy pressure            201
                       Latency spike likely driven by saturation and queue buildup      126
total_iops             High IOPS with small requests suggests random IO pressure        